### Bronze Layer (DLT)
Streaming Tables that incrementally pick up newly landed JSON files via Auto Loader. No transformation logic beyond capturing the raw payload plus file lineage metadata — kept append-only and schema-tolerant so a change in the API response never breaks ingestion.

Add this notebook, `2 - Silver (DLT)`, and `3 - Gold (DLT)` as the **source code** files for a single DLT pipeline. Set the pipeline's **Destination catalog** to `catalog` and **target schema** to `stocks` (pipeline writes are limited to one schema, so bronze/silver/gold all land in `catalog.stocks`, distinguished by table name prefix).

In [0]:
import dlt
from pyspark.sql import functions as F

LANDING_PATH = "/Volumes/catalog/bronze_landing"

In [0]:
@dlt.table(
    name="bronze_quotes",
    comment="Raw GLOBAL_QUOTE snapshots landed from the Alpha Vantage API, one row per file.",
    table_properties={"quality": "bronze"}
)
@dlt.expect_or_drop("has_quote_payload", "global_quote IS NOT NULL")
def bronze_quotes():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", f"{LANDING_PATH}/_schemas/quotes")
            .option("cloudFiles.inferColumnTypes", "true")
            .load(f"{LANDING_PATH}/quotes/")
            .select(
                F.col("`Global Quote`").alias("global_quote"),
                F.col("_metadata.file_path").alias("source_file"),
                F.col("_metadata.file_modification_time").alias("ingested_at")
            )
    )

In [0]:
@dlt.table(
    name="bronze_company_info",
    comment="Raw OVERVIEW payloads landed from the Alpha Vantage API, one row per file.",
    table_properties={"quality": "bronze"}
)
@dlt.expect_or_drop("has_symbol", "Symbol IS NOT NULL")
def bronze_company_info():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", f"{LANDING_PATH}/_schemas/company_info")
            .option("cloudFiles.inferColumnTypes", "true")
            .load(f"{LANDING_PATH}/company_info/")
            .select(
                "*",
                F.col("_metadata.file_path").alias("source_file"),
                F.col("_metadata.file_modification_time").alias("ingested_at")
            )
    )